# Doenças em Folhas de Plantas

Notebook para seminário acadêmico sobre classificação de doenças em folhas usando visão computacional.

Dataset: Plant Disease Recognition Dataset, disponível no Kaggle em `rashikrahmanpritom/plant-disease-recognition-dataset`.

O notebook executa EDA, pré-processamento, treinamento de uma CNN baseline, treinamento de MobileNetV2 com Transfer Learning, avaliação e exportação dos arquivos usados pela interface Streamlit.

**Importante:** as métricas devem ser obtidas ao executar o notebook. Nenhum resultado numérico é assumido ou inventado.

## 1. Instalação e imports

No Google Colab, TensorFlow normalmente já vem instalado. As demais bibliotecas abaixo são usadas para download, EDA e avaliação.

In [ ]:
!pip -q install kaggle scikit-learn seaborn

In [ ]:
import json
import os
import random
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from IPython.display import display
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

sns.set_theme(style='whitegrid')
print('TensorFlow:', tf.__version__)

## 2. Download do dataset

Para baixar pelo Kaggle no Colab:

1. Crie um token em Kaggle > Account > API > Create New Token.
2. Execute a célula abaixo e envie o arquivo `kaggle.json` quando solicitado.
3. O dataset será baixado para `data/raw/`.

Se preferir, baixe manualmente o dataset no Kaggle, extraia os arquivos em `data/raw/` e pule o trecho de download.

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
MODEL_DIR = PROJECT_DIR / 'models'

DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SLUG = 'rashikrahmanpritom/plant-disease-recognition-dataset'
USE_KAGGLE_DOWNLOAD = True

if USE_KAGGLE_DOWNLOAD and not any(RAW_DIR.rglob('*')):
    try:
        from google.colab import files

        kaggle_json = Path('/root/.kaggle/kaggle.json')
        if not kaggle_json.exists():
            print('Envie seu arquivo kaggle.json.')
            uploaded = files.upload()
            Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
            shutil.copy(next(iter(uploaded.keys())), kaggle_json)
            os.chmod(kaggle_json, 0o600)
    except ImportError:
        print('Ambiente local detectado. Garanta que o Kaggle CLI esteja configurado.')

    !kaggle datasets download -d {DATASET_SLUG} -p {RAW_DIR} --unzip
else:
    print('Download ignorado: data/raw/ ja contem arquivos ou USE_KAGGLE_DOWNLOAD=False.')

## 3. Localização dos splits

O dataset do Kaggle costuma vir separado em treino, validação e teste. A função abaixo localiza esses diretórios mesmo quando há uma pasta intermediária, como `Train/Train/Healthy`.

Caso o dataset esteja em uma única pasta com subpastas de classes, o notebook cria uma divisão estratificada em `data/processed/`.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def image_files(path: Path):
    return [file for file in path.rglob('*') if file.is_file() and file.suffix.lower() in IMAGE_EXTENSIONS]


def direct_class_dirs(path: Path):
    if not path.exists():
        return []
    return [child for child in path.iterdir() if child.is_dir() and image_files(child)]


def find_split_dir(root: Path, names: list[str]):
    names = {name.lower() for name in names}
    candidates = []
    for path in [root, *root.rglob('*')]:
        if path.is_dir() and path.name.lower() in names and len(direct_class_dirs(path)) >= 2:
            candidates.append(path)
    return sorted(candidates, key=lambda item: len(item.parts))[0] if candidates else None


def copy_split(files, destination_dir: Path):
    destination_dir.mkdir(parents=True, exist_ok=True)
    for index, source in enumerate(files):
        destination = destination_dir / f'{index:05d}_{source.name}'
        shutil.copy2(source, destination)


def create_stratified_splits(source_root: Path, output_root: Path):
    class_dirs = direct_class_dirs(source_root)
    if len(class_dirs) < 2:
        raise FileNotFoundError('Nao encontrei splits nem subpastas de classes validas em data/raw/.')

    if output_root.exists():
        shutil.rmtree(output_root)

    for class_dir in class_dirs:
        files = image_files(class_dir)
        train_val, test = train_test_split(files, test_size=0.10, random_state=SEED)
        train, val = train_test_split(train_val, test_size=0.1111, random_state=SEED)

        copy_split(train, output_root / 'train' / class_dir.name)
        copy_split(val, output_root / 'validation' / class_dir.name)
        copy_split(test, output_root / 'test' / class_dir.name)

    return output_root / 'train', output_root / 'validation', output_root / 'test'


TRAIN_DIR = find_split_dir(RAW_DIR, ['train', 'training'])
VAL_DIR = find_split_dir(RAW_DIR, ['val', 'valid', 'validation'])
TEST_DIR = find_split_dir(RAW_DIR, ['test', 'testing'])

if not all([TRAIN_DIR, VAL_DIR, TEST_DIR]):
    print('Splits prontos nao encontrados. Criando divisao treino/validacao/teste em data/processed/.')
    TRAIN_DIR, VAL_DIR, TEST_DIR = create_stratified_splits(RAW_DIR, DATA_DIR / 'processed')

print('Treino:', TRAIN_DIR)
print('Validacao:', VAL_DIR)
print('Teste:', TEST_DIR)

## 4. EDA das imagens

Nesta seção avaliamos quantidade de classes, quantidade de imagens por classe, exemplos visuais, resolução das imagens e possível desbalanceamento.

In [ ]:
split_dirs = {
    'train': TRAIN_DIR,
    'validation': VAL_DIR,
    'test': TEST_DIR,
}

records = []
for split_name, split_path in split_dirs.items():
    for class_dir in direct_class_dirs(split_path):
        for file in image_files(class_dir):
            records.append({
                'split': split_name,
                'class_name': class_dir.name,
                'path': str(file),
            })

df = pd.DataFrame(records)
if df.empty:
    raise RuntimeError('Nenhuma imagem foi encontrada. Verifique data/raw/.')

class_names = sorted(df[df['split'] == 'train']['class_name'].unique().tolist())
num_classes = len(class_names)

print(f'Quantidade de classes: {num_classes}')
print('Classes:', class_names)

class_counts = (
    df.groupby(['split', 'class_name'])
    .size()
    .reset_index(name='num_images')
    .sort_values(['split', 'class_name'])
)
display(class_counts)

plt.figure(figsize=(10, 5))
sns.barplot(data=class_counts, x='class_name', y='num_images', hue='split')
plt.title('Quantidade de imagens por classe e split')
plt.xlabel('Classe')
plt.ylabel('Quantidade de imagens')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

train_counts = class_counts[class_counts['split'] == 'train'].set_index('class_name')['num_images']
imbalance_ratio = train_counts.max() / train_counts.min()
print(f'Razao max/min no treino: {imbalance_ratio:.2f}')
if imbalance_ratio >= 1.5:
    print('Possivel desbalanceamento identificado. Considere pesos de classe ou mais dados.')
else:
    print('Nao ha indicio forte de desbalanceamento pelo criterio max/min >= 1.5.')

In [ ]:
samples = (
    df[df['split'] == 'train']
    .groupby('class_name', group_keys=False)
    .apply(lambda group: group.sample(min(3, len(group)), random_state=SEED))
)

rows = num_classes
cols = 3
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = np.array(axes).reshape(rows, cols)

for row_index, class_name in enumerate(class_names):
    class_samples = samples[samples['class_name'] == class_name]['path'].tolist()
    for col_index in range(cols):
        ax = axes[row_index, col_index]
        ax.axis('off')
        if col_index < len(class_samples):
            image = Image.open(class_samples[col_index]).convert('RGB')
            ax.imshow(image)
            ax.set_title(class_name)

plt.tight_layout()
plt.show()

In [ ]:
resolution_records = []
for image_path in df['path']:
    try:
        with Image.open(image_path) as image:
            width, height = image.size
        resolution_records.append({
            'path': image_path,
            'width': width,
            'height': height,
            'resolution': f'{width}x{height}',
        })
    except UnidentifiedImageError:
        print('Imagem invalida ignorada:', image_path)

resolution_df = pd.DataFrame(resolution_records)
display(resolution_df[['width', 'height']].describe())
display(resolution_df['resolution'].value_counts().head(10).rename('count'))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(resolution_df['width'], bins=20, ax=axes[0])
axes[0].set_title('Distribuicao das larguras')
sns.histplot(resolution_df['height'], bins=20, ax=axes[1])
axes[1].set_title('Distribuicao das alturas')
plt.tight_layout()
plt.show()

## 5. Pré-processamento

As imagens são redimensionadas para 224x224, normalizadas para pixels entre 0 e 1 e carregadas em `tf.data.Dataset`. O data augmentation é aplicado durante o treinamento com rotação, flip horizontal, zoom e brilho.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=class_names,
    color_mode='rgb',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
)

val_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=class_names,
    color_mode='rgb',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_raw = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    labels='inferred',
    label_mode='categorical',
    class_names=class_names,
    color_mode='rgb',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


def normalize_pixels(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels


train_ds = train_raw.map(normalize_pixels, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds = val_raw.map(normalize_pixels, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds = test_raw.map(normalize_pixels, num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)


def make_data_augmentation():
    return keras.Sequential(
        [
            layers.RandomRotation(0.08, fill_mode='nearest'),
            layers.RandomFlip('horizontal'),
            layers.RandomZoom(0.15),
            layers.RandomBrightness(0.20, value_range=(0.0, 1.0)),
        ],
        name='data_augmentation',
    )


class_names_path = MODEL_DIR / 'class_names.json'
with class_names_path.open('w', encoding='utf-8') as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)

print('Mapeamento de classes salvo em:', class_names_path)

## 6. Modelos

Serão comparados dois modelos:

- CNN simples como baseline.
- MobileNetV2 com pesos ImageNet e Transfer Learning.

Ambos usam os callbacks EarlyStopping, ReduceLROnPlateau e ModelCheckpoint.

In [ ]:
def make_callbacks(model_name: str):
    checkpoint_path = MODEL_DIR / f'{model_name}.keras'
    return [
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=5,
            restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=2,
            min_lr=1e-6,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=checkpoint_path,
            monitor='val_loss',
            save_best_only=True,
        ),
    ]


def build_baseline_cnn(num_classes: int):
    inputs = keras.Input(shape=(224, 224, 3))
    x = make_data_augmentation()(inputs)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='baseline_cnn')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


def build_mobilenetv2(num_classes: int):
    base_model = keras.applications.MobileNetV2(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3),
    )
    base_model.trainable = False

    inputs = keras.Input(shape=(224, 224, 3))
    x = make_data_augmentation()(inputs)
    x = layers.Rescaling(scale=2.0, offset=-1.0, name='mobilenetv2_preprocess')(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs, outputs, name='mobilenetv2_transfer')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model


EPOCHS_BASELINE = 20
EPOCHS_TRANSFER = 15

In [ ]:
baseline_model = build_baseline_cnn(num_classes)
baseline_model.summary()

history_baseline = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_BASELINE,
    callbacks=make_callbacks('baseline_cnn'),
)

In [ ]:
mobilenet_model = build_mobilenetv2(num_classes)
mobilenet_model.summary()

history_mobilenet = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TRANSFER,
    callbacks=make_callbacks('mobilenetv2_transfer'),
)

## 7. Avaliação

A avaliação usa accuracy, F1-score macro, classification report, matriz de confusão e exemplos de acertos e erros.

In [ ]:
def collect_predictions(model, dataset, keep_images=False):
    y_true_batches = []
    y_prob_batches = []
    image_batches = []

    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        y_prob_batches.append(probabilities)
        y_true_batches.append(np.argmax(labels.numpy(), axis=1))
        if keep_images:
            image_batches.append(images.numpy())

    y_true = np.concatenate(y_true_batches)
    y_prob = np.concatenate(y_prob_batches)

    if keep_images:
        images = np.concatenate(image_batches)
        return y_true, y_prob, images

    return y_true, y_prob


def evaluate_model(model_path: Path, model_name: str):
    model = keras.models.load_model(model_path)
    y_true, y_prob = collect_predictions(model, test_ds)
    y_pred = np.argmax(y_prob, axis=1)

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)

    print(f'\nModelo: {model_name}')
    print(f'Accuracy: {accuracy:.4f}')
    print(f'F1-score macro: {f1_macro:.4f}')
    print('\nClassification report:')
    print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Matriz de confusao - {model_name}')
    plt.xlabel('Predito')
    plt.ylabel('Real')
    plt.tight_layout()
    plt.show()

    return {
        'model_name': model_name,
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'model_path': str(model_path),
    }


baseline_metrics = evaluate_model(MODEL_DIR / 'baseline_cnn.keras', 'CNN baseline')
mobilenet_metrics = evaluate_model(MODEL_DIR / 'mobilenetv2_transfer.keras', 'MobileNetV2 Transfer Learning')

comparison_df = pd.DataFrame([baseline_metrics, mobilenet_metrics]).sort_values('f1_macro', ascending=False)
display(comparison_df)

best_model_path = Path(comparison_df.iloc[0]['model_path'])
final_model_path = MODEL_DIR / 'best_model.keras'
shutil.copy2(best_model_path, final_model_path)

with (MODEL_DIR / 'class_names.json').open('w', encoding='utf-8') as file:
    json.dump(class_names, file, ensure_ascii=False, indent=2)

print('Melhor modelo salvo em:', final_model_path)
print('Classes salvas em:', MODEL_DIR / 'class_names.json')

In [ ]:
best_model = keras.models.load_model(MODEL_DIR / 'best_model.keras')
y_true, y_prob, test_images = collect_predictions(best_model, test_ds, keep_images=True)
y_pred = np.argmax(y_prob, axis=1)

correct_indices = np.where(y_true == y_pred)[0]
wrong_indices = np.where(y_true != y_pred)[0]
rng = np.random.default_rng(SEED)


def plot_prediction_examples(indices, title, max_examples=6):
    if len(indices) == 0:
        print(f'Nenhum exemplo para: {title}')
        return

    selected = rng.choice(indices, size=min(max_examples, len(indices)), replace=False)
    cols = 3
    rows = int(np.ceil(len(selected) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis('off')

    for ax, index in zip(axes, selected):
        confidence = y_prob[index, y_pred[index]]
        ax.imshow(test_images[index])
        ax.set_title(
            f'Real: {class_names[y_true[index]]}\nPred: {class_names[y_pred[index]]} ({confidence:.1%})'
        )

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_prediction_examples(correct_indices, 'Exemplos de acertos')
plot_prediction_examples(wrong_indices, 'Exemplos de erros')

## 8. Uso na interface Streamlit

Ao final da execução, baixe ou mantenha os arquivos abaixo na pasta `models/` do projeto local:

- `best_model.keras`
- `class_names.json`

Depois, execute localmente:

```bash
streamlit run app.py
```